# Composable plotting

The `bettermdptools.plotting` API separates pure data preparation from renderers that draw on explicit Matplotlib axes. The notebook owns every figure: save before `plt.show()` when needed, then close it explicitly.

In [ ]:
!pip install bettermdptools

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt

from bettermdptools.algorithms.planner import Planner
from bettermdptools.algorithms.rl import RL
from bettermdptools.envs.blackjack_wrapper import BlackjackWrapper
from bettermdptools.plotting import (
    plot_convergence,
    plot_learning_curve,
    plot_policy_grid,
    plot_value_convergence,
    plot_value_policy,
    prepare_convergence,
    prepare_learning_curve,
    prepare_policy_grid,
)

## 1. Episode reward learning curves

A one-dimensional reward array is one run. A two-dimensional array uses `(runs, episodes)`. Smoothing happens within each run before the center and cross-run quantile band are calculated.

In [ ]:
reward_runs = []
for seed in range(3):
    training_env = gym.make("FrozenLake-v1", is_slippery=False)
    try:
        *_, rewards = RL(training_env).q_learning(n_episodes=300, seed=seed)
        reward_runs.append(rewards)
    finally:
        training_env.close()

learning_curve = prepare_learning_curve(reward_runs, window=25)
fig, ax = plt.subplots(layout="constrained")
plot_learning_curve(
    learning_curve, ax=ax, title="FrozenLake Q-learning across three runs"
)
plt.show()
plt.close(fig)

## 2. Value and policy convergence

Model-free histories contain one valid entry per episode, including legitimate all-zero entries. `prepare_convergence` never guesses validity by trimming zeros.

In [ ]:
diagnostic_env = gym.make("FrozenLake-v1", is_slippery=False)
try:
    Q, _, _, Q_track, pi_track, _ = RL(diagnostic_env).q_learning(
        n_episodes=300, seed=7
    )
finally:
    diagnostic_env.close()

convergence = prepare_convergence(Q_track, policy_history=pi_track)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
plot_convergence(
    convergence,
    value_ax=axes[0],
    policy_ax=axes[1],
    title="FrozenLake Q-learning convergence",
)
plt.show()
plt.close(fig)

## Planner history validity

Planning histories retain their historical fixed allocation. Request metadata to identify the exact valid prefix, including a valid all-zero initial row.

In [ ]:
frozen_lake = gym.make("FrozenLake8x8-v1", render_mode=None)
V, V_track, pi, planning = Planner(
    frozen_lake.unwrapped.P
).value_iteration(n_iters=5000, return_metadata=True)
planner_convergence = prepare_convergence(
    V_track, valid_length=planning.history_length
)
fig, ax = plt.subplots(layout="constrained")
plot_value_convergence(
    planner_convergence, ax=ax, title="FrozenLake value iteration convergence"
)
plt.show()
plt.close(fig)

## 3. Value plus policy composition

Preparation validates state coverage and preserves complete action labels. Both renderers use the same numeric scale and only draw on the supplied axes.

In [ ]:
fl_actions = {0: "←", 1: "↓", 2: "→", 3: "↑"}
policy_grid = prepare_policy_grid(pi, V, fl_actions, (8, 8))
fig = plt.figure(figsize=(12, 5), layout="constrained")
grid = fig.add_gridspec(1, 3, width_ratios=(1, 1, 0.05))
value_ax = fig.add_subplot(grid[0, 0])
policy_ax = fig.add_subplot(grid[0, 1])
colorbar_ax = fig.add_subplot(grid[0, 2])
plot_value_policy(
    policy_grid,
    value_ax=value_ax,
    policy_ax=policy_ax,
    cbar_ax=colorbar_ax,
    value_title="FrozenLake state values",
    policy_title="FrozenLake policy",
)
# Save before a blocking show when output is needed:
# fig.savefig("frozen-lake-summary.png", dpi=150)
plt.show()
plt.close(fig)

### Documented style controls

Blackjack includes a terminal bust sink that is not part of the `(29, 10)` decision surface. Select decision states explicitly, then customize the public renderer instead of monkey-patching `Plots`.

In [ ]:
base_env = gym.make("Blackjack-v1", render_mode=None)
blackjack = BlackjackWrapper(base_env)
V_blackjack, _, pi_blackjack = Planner(blackjack.P).value_iteration()
decision_values = V_blackjack[:-1]
decision_policy = {
    state: pi_blackjack[state] for state in range(len(decision_values))
}
blackjack_grid = prepare_policy_grid(
    decision_policy, decision_values, {0: "STICK", 1: "HIT"}, (29, 10)
)
fig, ax = plt.subplots(figsize=(10, 12), layout="constrained")
plot_policy_grid(
    blackjack_grid,
    ax=ax,
    title="Blackjack decision policy",
    cmap="magma_r",
    annotation_kws={"fontsize": 7},
)
plt.show()
plt.close(fig)
frozen_lake.close()
blackjack.close()